WITH/WITHOUT BAROMETRIC ALTITUDE

In [ ]:
import re
import json

# === JSON cleanup ===
def clean_answer(raw_response: str):
    """
    Cleans and safely parses LLM JSON output (even if wrapped in markdown fences).
    """
    # Remove Markdown code fences (```json ... ```)
    cleaned = re.sub(r"^```(?:json|python)?", "", raw_response.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r"```$", "", cleaned.strip())

    # Remove common markdown artifacts
    cleaned = cleaned.replace("\n", "\\n").replace("\r", "\\r")

    # Some models may prepend language tag or commentary
    cleaned = re.sub(r"^json", "", cleaned.strip(), flags=re.IGNORECASE)

    # Extra cleanup for accidental trailing commas
    cleaned = re.sub(r",\s*([\]\}])", r"\1", cleaned)

    # Try JSON parsing
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        # print(f"⚠️ JSON parsing failed, attempting manual recovery: {e}")
        # # Attempt a secondary cleaning
        cleaned2 = cleaned.replace('\\n', ' ').replace('\\"', '"')
        return json.loads(cleaned2)

In [ ]:
import json
from openai import OpenAI
import os
import faiss
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path

# === Configuration ===
LLM_MODEL = "gpt-4.1"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY_2"))
PHASES = ["take_off", "cruising", "landing"]

# === Prompt de résumé phase (tu l’utilises déjà) ===
PROMPT_PHASE_TEMPLATE = """
You are given a segment of flight data corresponding to the {phase_name} phase of a flight. The data is structured as a list of datapoints, 
each containing the aircraft’s state and environmental conditions at a given time.

Each datapoint is a dictionary with keys such as:
- lat, lon, track, altitude_ft, gspeed (knots), vspeed (ft/min), minutes_since_start (float)
- edr_300hPa, ellrod_300hPa, richardson_number_300hPa, cape,
- wind_speed_10000m, wind_dir_10000m, wind_speed_2m, wind_dir_2m.

Your task is to write a **continuous, expert-level narrative** describing this phase of flight, 
progressing through the datapoints chronologically.

For each datapoint:
1. Use the field `minutes_since_start` to reference time relative to takeoff.
   Example: “5 minutes after departure” (for takeoff) or “5 minutes into the flight” (for cruise/landing).
2. Describe the aircraft's position (lat/lon) and motion (heading, altitude, ground speed, vertical speed).
3. Include meteorological context: turbulence (EDR, Ellrod), stability (Richardson number), CAPE, and wind information.
4. Mention relevant geographic regions based on coordinates (e.g., cities, monuments, regions, landscapes, montains, seas, rivers etc...).
5. Use **smooth transitions** between datapoints:
   - “Then the aircraft turns east by 20° and climbs another 300 ft...”
   - “A few minutes later, it begins a gentle descent...”
   - “As it approaches the Tyrrhenian coast...”
6. Do not forget to include the plane's type at the begining of your description: {type}

Write the result as a natural, continuous, and professional description, not as bullet points. I want to description to be very detailed.

Here is the flight phase's record:
{phase_data}

Return the flights summary without any metatext"""

# === Génère le résumé via OpenAI pour n points de données ===
def generate_summary(phase_name, phase_data, aircraft_type):
    prompt = PROMPT_PHASE_TEMPLATE.format(
        phase_name=phase_name,
        phase_data=json.dumps(phase_data),
        type=aircraft_type
    )
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    parsed = response.choices[0].message.content.strip()
    return parsed # narrative only


# === Construit le prompt final de prédiction ===
def build_prediction_prompt(target_summary, similar_summaries, N):
    prompt = f"""The target flight entered a GPS spoofed zone. Here is its trajectory summary before spoofing:
{target_summary}

Here are the summaries of the 5 most similar flights:"""
    for i, summ in enumerate(similar_summaries):
        prompt += f"\n\nSimilar flight {i+1}:\n{summ}"

    prompt += f"""

Based on these flights, where will the target flight be {N} minutes after the start of the phase?

⚠️ Return only the predicted position in this exact format:
LATITUDE, LONGITUDE
"""
    return prompt

# === Appel LLM pour la prédiction ===
def predict_position(prompt):
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()


In [ ]:
import os
import json
import faiss
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY_2"))

# --------------------------
# CONFIGURATION
# --------------------------
EMBED_MODEL = "text-embedding-3-large"
LLM_MODEL = "gpt-4.1"
K = 5  # number of similar flights retrieved

# --------------------------
# LOAD DATA
# --------------------------
with open("../../data/CDG-FCO/flight_data_with_minutes_since_start.json", "r", encoding="utf-8") as f:
    flights_data = json.load(f)

with open("../../data/CDG-FCO/flight_summary_separated_into_phases.json", "r", encoding="utf-8") as f:
    summaries = json.load(f)

data_by_id = {f["flight_metadata"]["fr24_id"]: f for f in flights_data}
summary_by_phase = {
    s["fr24_id"]: {p: s.get(p, None) for p in ["take_off", "cruising", "landing"]}
    for s in summaries
}

print(f"✅ Loaded {len(flights_data)} flights with data and summaries.\n")

def load_phase_index(phase):
    index = faiss.read_index(f"../../data/CDG-FCO/EMBEDDINGS/faiss_index_{phase}.index")
    embeddings = np.load(f"../../data/CDG-FCO/EMBEDDINGS/embeddings_{phase}.npy")
    with open(f"../../data/CDG-FCO/EMBEDDINGS/ids_{phase}.json", "r") as f:
        ids = json.load(f)
    print(f"📂 Loaded FAISS index, embeddings and IDs for phase '{phase}'")
    return index, ids, embeddings

# --------------------------
# EMBEDDING HELPER
# --------------------------
def get_embeddings(texts, model=EMBED_MODEL, batch_size=32):
    """Embed multiple texts with normalization."""
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        resp = client.embeddings.create(model=model, input=batch)
        embs = [np.array(e.embedding, dtype="float32") for e in resp.data]
        vecs.extend(embs)
    X = np.vstack(vecs)
    X /= np.linalg.norm(X, axis=1, keepdims=True)
    return X

# --------------------------
# LOAD RAG INDEX FOR EACH PHASE
# --------------------------

def build_phase_index(phase):
    return load_phase_index(phase)

In [ ]:
phase_index_take_off, phase_ids_take_off, _ = build_phase_index("take_off")
phase_index_cruising, phase_ids_cruising, _ = build_phase_index("cruising")
phase_index_landing, phase_ids_landing, _ = build_phase_index("landing")

In [ ]:
index_dictionary = {"take_off": phase_index_take_off, "cruising": phase_index_cruising, "landing": phase_index_landing}
ids_dictionary = {"take_off": phase_ids_take_off, "cruising": phase_ids_cruising, "landing": phase_ids_landing}

In [ ]:
PREDICT_POSITION_PROMPT = f"""
A flight entered a GPS spoofing zone. Therefore his position is now unknown and your task is to help the pilot 
by predicting the flight's latitude and longitude.
Here is its known trajectory (before spoofing):

{{partial_summary}}

The following are summaries of {{number_similar_flight}} similar flights for the same phase:

{{sim_text}}

{{predictions_history}}

Based on these, predict the geographic position of the target flight {{predict_minutes}} minutes after the beginning of the phase.
Return your answer as JSON: {{{{"lat": "the predicted latitude", "lon": "the predicted longitude"}}}} only.
"""

# --------------------------
# PREDICT POSITION VIA LLM
# --------------------------
def predict_position_from_prompt(partial_summary, similar_summaries, predict_minutes, prediction_history: list = []):
    """Ask GPT to predict target position after N minutes using similar flights."""
    sim_text = "\n\n".join([f"Similar flight {i+1}:\n{txt}" for i, txt in enumerate(similar_summaries)])
    if not prediction_history:
        predictions_history = ""
    else:
        predictions_history = f"""Here are the predictions you already made. They are given as a list of triplet [latitude, longitude, number of minutes into the phase]. Use these past predictions to ensure that your next prediction is consistent.
Make sure the plane continues moving forward along its trajectory, does not remain at the same coordinates repeatedly, and does not move backward in a way that violates realistic flight dynamics.

PREDICTION HISTORY: {{prediction_history}}"""
        predictions_history = predictions_history.format(prediction_history=prediction_history)

    prompt = PREDICT_POSITION_PROMPT.format(
        partial_summary=partial_summary, 
        number_similar_flight=len(similar_summaries),
        sim_text=sim_text,
        predict_minutes=predict_minutes,
        predictions_history=predictions_history,
    )
        
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return clean_answer(response.choices[0].message.content), response.usage.total_tokens


In [ ]:
# --------------------------
# FIND SIMILAR FLIGHTS
# --------------------------
def find_similar_flights(partial_summary, target_id, phase, k=K):
    """Embed the target partial summary and find K most similar full summaries."""
    q_vec = get_embeddings([partial_summary])
    sims, idxs = index_dictionary[phase].search(q_vec, k)
    ids_for_phase = ids_dictionary[phase]
    sims, idxs = sims[0], idxs[0]
    neighbors = [(ids_for_phase[i], float(sims[i])) for i in range(len(idxs)) if ids_for_phase[i] != target_id]
    return neighbors


In [ ]:
import math

def haversine(lat1, lon1, lat2, lon2):
    R = 6371e3
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

def predict_flight(flight_id, phase, with_feedback_loop=True, k=K):
    # --------------------------
    # 1️⃣ Extraire les données du vol cible
    # --------------------------
    target_flight = data_by_id[flight_id]
    phase_data = target_flight[phase]
    aircraft_type = target_flight["flight_metadata"]["type"]

    # --------------------------
    # 2️⃣ Définir la zone de spoofing
    # --------------------------
    N = len(phase_data)
    SPOOF_INDEX = N // 2   # floor division → produit exactement la séparation souhaitée

    # print(f"📏 Phase length = {N} points — Observed = 0:{SPOOF_INDEX-1}, Predicted = {SPOOF_INDEX}:{N-1}")

    observed_data = phase_data[:SPOOF_INDEX]
    spoofed_data  = phase_data[SPOOF_INDEX:]

    # --------------------------
    # 3️⃣ Générer le résumé partiel (RAG query)
    # --------------------------
    partial_summary = generate_summary(phase, observed_data, aircraft_type)
    # print("🧩 Résumé partiel généré.\n")

    # --------------------------
    # 5️⃣ Prédire tous les points de la zone spoofée
    # --------------------------
    predicted_positions = []
    ground_truth_positions = []

    for i, point in enumerate(spoofed_data):
        t_min = point["minutes_since_start"]
        # --------------------------
        # 4️⃣ Trouver les vols similaires en prenant en compte l'altitude connu
        # --------------------------
        neighbors = find_similar_flights(partial_summary, target_id=flight_id, phase=phase, k=k)
        similar_ids = [fid for fid, _ in neighbors]
        similar_summaries = [summary_by_phase[fid][phase] for fid in similar_ids]

        # print(f"🔍 {len(similar_ids)} vols similaires trouvés :", similar_ids, "\n")

        # print(f"⏳ Prédiction point {SPOOF_INDEX+i} (t = {t_min} min)")

        if with_feedback_loop:
            prediction, _ = predict_position_from_prompt(
                partial_summary,
                similar_summaries,
                predict_minutes=t_min,
                prediction_history=predicted_positions
            )
        else:
            prediction, _ = predict_position_from_prompt(
                partial_summary,
                similar_summaries,
                predict_minutes=t_min
            )

        if prediction:
            predicted_positions.append([float(prediction["lat"]), float(prediction["lon"]), t_min]) # 
        else:
            predicted_positions.append([None, None, t_min])

        ground_truth_positions.append([point["lat"], point["lon"], t_min])

    mean_error = 0

    for i in range(len(predicted_positions)):
        predicted_point = predicted_positions[i]
        lat1 = predicted_point[0]
        lon1 = predicted_point[1]
        ground_truth_point = ground_truth_positions[i]
        lat2 = ground_truth_point[0]
        lon2 = ground_truth_point[1]
        error = haversine(lat1, lon1, lat2, lon2)
        mean_error += error

    mean_error = mean_error / len(predicted_positions)

    return round(mean_error/1000, 1)


In [ ]:
import json

with open("../../data/CDG-FCO/RESULTS/test_sample.json", "r", encoding="utf-8") as f:
    flight_ids_by_phase = json.load(f)

results = {"take_off": 0, "cruising": 0, "landing": 0}

In [ ]:
import random

def calculate_mean_haversine_error(
    flight_ids_by_phase,
    k=5,
):
    errors_takeoff = []
    errors_cruising = []
    errors_landing = []

    counter_takeoff = 0
    counter_cruising = 0
    counter_landing = 0

    # ======================================================
    # TAKE OFF
    # ======================================================
    for flight_id in flight_ids_by_phase["take_off"]:
        error= predict_flight(
            flight_id, "take_off",
            with_feedback_loop=True,
            k=k,
        )
        errors_takeoff.append((error, flight_id))
        counter_takeoff += 1
        print(f"counter take off: {counter_takeoff}")

    # ======================================================
    # CRUISING
    # ======================================================
    for flight_id in flight_ids_by_phase["cruising"]:
        error = predict_flight(
            flight_id, "cruising",
            with_feedback_loop=True,
            k=k,
        )
        errors_cruising.append((error, flight_id))
        counter_cruising += 1
        print(f"counter cruising: {counter_cruising}")
    

    # ======================================================
    # LANDING
    # ======================================================
    for flight_id in flight_ids_by_phase["landing"]:
        error = predict_flight(
            flight_id, "landing",
            with_feedback_loop=True,
            k=k,
        )
        errors_landing.append((error, flight_id))
        counter_landing += 1
        print(f"counter landing: {counter_landing}")

    # ======================================================
    # Means
    # ======================================================

    return (
        errors_takeoff,
        errors_cruising,
        errors_landing,
    )


In [ ]:
import json

output_file = "../with_without_barometric_altitude/HAVERSINE_ERROR_RAG_WITHOUT_BAROMETRIC_ALTITUDE.json"

takeoff, cruising, landing = calculate_mean_haversine_error(flight_ids_by_phase, k=5)

results = {
    "RAG": 
           {"take_off": takeoff, 
            "cruising": cruising, 
            "landing": landing}
}

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print(f"✅ Saved to {output_file}")

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

# =========================
# Paths
# =========================
PATH_RAG = "../../data/CDG-FCO/RESULTS/MEAN_HAVERSINE_RAG.json"
PATH_RAG_NO_BARO = "../with_without_barometric_altitude/HAVERSINE_ERROR_RAG_WITHOUT_BAROMETRIC_ALTITUDE.json"

PHASES = ["take_off", "cruising", "landing"]

# =========================
# Load data
# =========================
with open(PATH_RAG, "r", encoding="utf-8") as f:
    rag_data = json.load(f)["RAG"]

with open(PATH_RAG_NO_BARO, "r", encoding="utf-8") as f:
    rag_no_baro_raw = json.load(f)["RAG"]

# Extract only the error value (index 0) from [error, flight_id]
rag_no_baro_data = {
    phase: [v[0] for v in rag_no_baro_raw[phase]]
    for phase in PHASES
}

# =========================
# Prepare boxplot data
# =========================
box_data = []
positions = []

pos = 1
gap_between_phases = 1.6

for phase in PHASES:
    box_data.append(rag_data[phase])
    box_data.append(rag_no_baro_data[phase])

    positions.extend([pos, pos + 0.6])
    pos += gap_between_phases + 0.6

# =========================
# Plot
# =========================
plt.figure(figsize=(11, 6))

bp = plt.boxplot(
    box_data,
    positions=positions,
    widths=0.5,
    patch_artist=True,
    showmeans=True
)

# Colors: same color per method across phases
colors = ["#4C72B0", "#DD8452"] * len(PHASES)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)

# Mean marker styling
for mean in bp["means"]:
    mean.set_marker("o")
    mean.set_markerfacecolor("black")
    mean.set_markeredgecolor("black")

# =========================
# X-axis (phase-centered)
# =========================
phase_centers = [
    np.mean(positions[i * 2:(i * 2) + 2])
    for i in range(len(PHASES))
]

plt.xticks(phase_centers, ["Take-off", "Cruising", "Landing"])
plt.ylabel("Haversine error (km)")
plt.title("Ablation Study – Effect of Removing Barometric Altitude in RAG Retrieval")

# Legend
plt.plot([], [], color="#4C72B0", label="RAG")
plt.plot([], [], color="#DD8452", label="RAG without barometric altitude")
plt.legend()

plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
import json
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

# =========================
# Paths
# =========================
PATH_RAG = "../../data/CDG-FCO/RESULTS/MEAN_HAVERSINE_RAG.json"
PATH_RAG_NO_BARO = "../with_without_barometric_altitude/HAVERSINE_ERROR_RAG_WITHOUT_BAROMETRIC_ALTITUDE.json"
OUTPUT_CSV = "../with_without_barometric_altitude/RAG_BAROMETRIC_ABLATION_STATS.csv"

In [ ]:
PHASES = ["take_off", "cruising", "landing"]

# =========================
# Load data
# =========================
with open(PATH_RAG, "r", encoding="utf-8") as f:
    rag_data = json.load(f)["RAG"]

with open(PATH_RAG_NO_BARO, "r", encoding="utf-8") as f:
    rag_no_baro_raw = json.load(f)["RAG"]

# Extract only error values from [error, flight_id]
rag_data = {p: np.array(rag_data[p]) for p in PHASES}
rag_no_baro_data = {
    p: np.array([v[0] for v in rag_no_baro_raw[p]])
    for p in PHASES
}

# =========================
# Helper: descriptive stats
# =========================
def describe(values):
    return {
        "mean": np.mean(values),
        "median": np.median(values),
        "std": np.std(values, ddof=1),
        "q1": np.percentile(values, 25),
        "q3": np.percentile(values, 75),
        "min": np.min(values),
        "max": np.max(values),
        "n": len(values),
    }

def round_stats(d, decimals=2):
    out = {}
    for k, v in d.items():
        if isinstance(v, (float, np.floating)):
            out[k] = round(v, decimals)
        else:
            out[k] = v
    return out

# =========================
# Collect stats
# =========================
rows = []

for phase in PHASES:
    for method, values in [
        ("RAG", rag_data[phase]),
        ("RAG_no_barometric", rag_no_baro_data[phase]),
    ]:
        stats = describe(values)
        stats = round_stats(stats, decimals=2)

        rows.append({
            "phase": phase,
            "method": method,
            **stats
        })

# =========================
# Paired ablation statistics
# =========================
for phase in PHASES:
    diff = rag_no_baro_data[phase] - rag_data[phase]

    _, p_value = wilcoxon(
        rag_data[phase],
        rag_no_baro_data[phase],
        alternative="two-sided"
    )

    stats = describe(diff)
    stats["wilcoxon_p_value"] = p_value
    stats = round_stats(stats, decimals=2)

    rows.append({
        "phase": phase,
        "method": "ABLATION_COMPARISON",
        **stats
    })

# =========================
# Save CSV
# =========================
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

print(f"✅ Statistical results written to:\n{OUTPUT_CSV}")
